#### train_ml.py
##### ML baseline
- Load shared dataset + shared split
- TF-IDF
- SMOTE
- Train 4 ML models
- Save best ML model

In [1]:
import os
import json
import joblib
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE

### Files

In [2]:
SHARED_DATASET_FILE = "../data/processed/cleaned_emotions_shared.csv"
SHARED_SPLIT_FILE = "../data/splits/shared_split.json"

RESULT_FILE = "../data/processed/ml_all_model_results.csv"
PRED_FILE = "../data/processed/ml_test_predictions.csv"

MODELS_DIR = "../models"
ALL_MODELS_DIR = "../models/ml_models"

BEST_MODEL_FILE = "../models/ml_best_model.joblib"
BEST_VECTORIZER_FILE = "../models/ml_tfidf_vectorizer.joblib"
BEST_METADATA_FILE = "../models/ml_best_metadata.json"

ALL_MODELS_METADATA_FILE = "../models/ml_all_models_metadata.json"

RANDOM_STATE = 42

os.makedirs("../models", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)
os.makedirs(ALL_MODELS_DIR, exist_ok=True)

### Load shared data

In [3]:
df = pd.read_csv(SHARED_DATASET_FILE)

with open(SHARED_SPLIT_FILE, "r", encoding="utf-8") as f:
    split_dict = json.load(f)

train_df = df[df["row_id"].isin(split_dict["train_ids"])].copy()
val_df = df[df["row_id"].isin(split_dict["val_ids"])].copy()
test_df = df[df["row_id"].isin(split_dict["test_ids"])].copy()

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

# ML ใช้ train + val
train_full_df = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)

X_train_text = train_full_df["text"]
y_train = train_full_df["label"]

X_test_text = test_df["text"]
y_test = test_df["label"]


Train: (1512, 3)
Val  : (324, 3)
Test : (324, 3)


### TF-IDF

In [4]:
vectorizer = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    min_df=1
)

X_train_tfidf = vectorizer.fit_transform(X_train_text)
X_test_tfidf = vectorizer.transform(X_test_text)

print("\nTF-IDF shapes")
print("Train:", X_train_tfidf.shape)
print("Test :", X_test_tfidf.shape)



TF-IDF shapes
Train: (1836, 3000)
Test : (324, 3000)


### SMOTE

In [5]:
X_train_dense = X_train_tfidf.toarray()
X_test_dense = X_test_tfidf.toarray()

smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train_dense, y_train)

print("\nAfter SMOTE:", X_train_smote.shape)
print(pd.Series(y_train_smote).value_counts())

c:\Users\DELL\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\DELL\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\DELL\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\DELL\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^


After SMOTE: (2550, 3000)
label
angry      425
disgust    425
happy      425
fear       425
sad        425
neutral    425
Name: count, dtype: int64


### Models

In [6]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Support Vector Machine": SVC(kernel="linear", probability=True, random_state=RANDOM_STATE),
    "Naive Bayes": MultinomialNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
}

all_results = []
all_outputs = {}

def evaluate_model(model, X_train, y_train, X_test, y_test, model_name):
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average="macro")
    f1_weighted = f1_score(y_test, y_pred, average="weighted")

    print(f"\n===== {model_name} =====")
    print(f"Accuracy      : {acc:.4f}")
    print(f"F1-score Macro: {f1_macro:.4f}")
    print(f"F1-score Weighted: {f1_weighted:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    return {
        "model": model,
        "model_name": model_name,
        "y_pred": y_pred,
        "y_proba": y_proba,
        "Accuracy": acc,
        "F1_Macro": f1_macro,
        "F1_Weighted": f1_weighted,
        "classes": list(model.classes_)
    }

for model_name, model in models.items():
    result = evaluate_model(
        model,
        X_train_smote,
        y_train_smote,
        X_test_dense,
        y_test,
        model_name
    )

    all_results.append({
        "Model": result["model_name"],
        "Accuracy": result["Accuracy"],
        "F1_Macro": result["F1_Macro"],
        "F1_Weighted": result["F1_Weighted"]
    })

    all_outputs[model_name] = result


===== Logistic Regression =====
Accuracy      : 0.9074
F1-score Macro: 0.9229
F1-score Weighted: 0.9075

Confusion Matrix:
[[49  0  0  0  0  0]
 [ 0 59  1  0  4  0]
 [ 0  4 49  0  7  0]
 [ 0  0  0 34  1  0]
 [ 0  5  7  0 63  0]
 [ 0  0  1  0  0 40]]

Classification Report:
              precision    recall  f1-score   support

       angry       1.00      1.00      1.00        49
     disgust       0.87      0.92      0.89        64
        fear       0.84      0.82      0.83        60
       happy       1.00      0.97      0.99        35
     neutral       0.84      0.84      0.84        75
         sad       1.00      0.98      0.99        41

    accuracy                           0.91       324
   macro avg       0.93      0.92      0.92       324
weighted avg       0.91      0.91      0.91       324


===== Support Vector Machine =====
Accuracy      : 0.9228
F1-score Macro: 0.9355
F1-score Weighted: 0.9229

Confusion Matrix:
[[49  0  0  0  0  0]
 [ 0 58  2  0  4  0]
 [ 0  3 51  0

### Summary

In [7]:
results_df = pd.DataFrame(all_results).sort_values(by="F1_Macro", ascending=False)

print("\n===== ML Summary =====")
print(results_df)

results_df.to_csv(RESULT_FILE, index=False, encoding="utf-8-sig")


===== ML Summary =====
                    Model  Accuracy  F1_Macro  F1_Weighted
1  Support Vector Machine  0.922840  0.935528     0.922900
0     Logistic Regression  0.907407  0.922935     0.907498
3           Random Forest  0.876543  0.897599     0.876845
2             Naive Bayes  0.867284  0.880318     0.863803


### Save best model

In [8]:
results_df = pd.DataFrame(all_results).sort_values(by="F1_Macro", ascending=False)

print("\n===== ML Summary =====")
print(results_df)

results_df.to_csv(RESULT_FILE, index=False, encoding="utf-8-sig")


===== ML Summary =====
                    Model  Accuracy  F1_Macro  F1_Weighted
1  Support Vector Machine  0.922840  0.935528     0.922900
0     Logistic Regression  0.907407  0.922935     0.907498
3           Random Forest  0.876543  0.897599     0.876845
2             Naive Bayes  0.867284  0.880318     0.863803


### Save shared vectorizer

In [9]:
joblib.dump(vectorizer, BEST_VECTORIZER_FILE)
print("\nSaved vectorizer:", BEST_VECTORIZER_FILE)


Saved vectorizer: ../models/ml_tfidf_vectorizer.joblib


### Save shared vectorizer

In [10]:
all_models_metadata = []

model_file_names = {
    "Logistic Regression": "logistic_regression.joblib",
    "Support Vector Machine": "svm.joblib",
    "Naive Bayes": "naive_bayes.joblib",
    "Random Forest": "random_forest.joblib"
}

for model_name, result in all_outputs.items():
    model_file = os.path.join(ALL_MODELS_DIR, model_file_names[model_name])
    joblib.dump(result["model"], model_file)

    all_models_metadata.append({
        "model_name": model_name,
        "file_path": model_file,
        "classes": result["classes"],
        "Accuracy": result["Accuracy"],
        "F1_Macro": result["F1_Macro"],
        "F1_Weighted": result["F1_Weighted"]
    })

    print("Saved model:", model_name, "->", model_file)

with open(ALL_MODELS_METADATA_FILE, "w", encoding="utf-8") as f:
    json.dump(all_models_metadata, f, ensure_ascii=False, indent=2)

print("\nSaved all models metadata:", ALL_MODELS_METADATA_FILE)

Saved model: Logistic Regression -> ../models/ml_models\logistic_regression.joblib
Saved model: Support Vector Machine -> ../models/ml_models\svm.joblib
Saved model: Naive Bayes -> ../models/ml_models\naive_bayes.joblib
Saved model: Random Forest -> ../models/ml_models\random_forest.joblib

Saved all models metadata: ../models/ml_all_models_metadata.json


### Save predictions of all 4 models

In [11]:
pred_df = test_df[["row_id", "text", "label"]].copy()

for model_name, result in all_outputs.items():
    short_name_map = {
        "Logistic Regression": "lr",
        "Support Vector Machine": "svm",
        "Naive Bayes": "nb",
        "Random Forest": "rf"
    }

    short_name = short_name_map[model_name]

    pred_df[f"{short_name}_pred"] = result["y_pred"]
    pred_df[f"{short_name}_confidence"] = result["y_proba"].max(axis=1)

    proba_df = pd.DataFrame(
        result["y_proba"],
        columns=[f"{short_name}_proba_{c}" for c in result["classes"]]
    )

    pred_df = pd.concat([pred_df.reset_index(drop=True), proba_df.reset_index(drop=True)], axis=1)

pred_df.to_csv(PRED_FILE, index=False, encoding="utf-8-sig")
print("\nSaved ML predictions of all 4 models:", PRED_FILE)


Saved ML predictions of all 4 models: ../data/processed/ml_test_predictions.csv
